In [6]:
import pandas as pd
import numpy as np

#Load raw dataset
df_raw = pd.read_csv('production.csv')
print("--- Raw Dataset Shape ---")
print(df_raw.shape)

print("\n--- Initial Schema & Data Types ---")
df_raw.info()

print("\n--- Missing Values Audit ---")
print(df_raw.isnull().sum())

print("\n--- Descriptive Summary of Numeric Fields ---")
print(df_raw.describe())

--- Raw Dataset Shape ---
(452, 13)

--- Initial Schema & Data Types ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 452 entries, 0 to 451
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Mineral              452 non-null    object 
 1   States               452 non-null    object 
 2   Unit                 452 non-null    object 
 3   Quantity 2010-11     444 non-null    float64
 4   Value 2010-11        444 non-null    float64
 5   Quantity 2011-12     442 non-null    float64
 6   Value 2011-12        442 non-null    float64
 7   Quantity 2012-13     443 non-null    float64
 8   Value 2012-13        443 non-null    float64
 9   Quantity 2013-14     447 non-null    float64
 10  Value 2013-14        448 non-null    float64
 11  Quantity 2014-15(P)  442 non-null    float64
 12  Value 2014-15(P)     442 non-null    float64
dtypes: float64(10), object(3)
memory usage: 46.0+ KB

--- Missing Value

In [7]:
df_coal = df_raw[df_raw['Mineral'] == 'Coal'].dropna(subset=['States']).copy()
df_states = df_coal[df_coal['States'].str.strip() != 'India'].copy()
print(f"Total State Records: {len(df_states)}")

Total State Records: 13


In [8]:
# Unpivot Quantity and Value columns
qty_cols = [c for c in df_states.columns if 'Quantity' in c]
val_cols = [c for c in df_states.columns if 'Value' in c]

# Melt Quantity
df_qty = pd.melt(
    df_states,
    id_vars=['States', 'Unit'],
    value_vars=qty_cols,
    var_name='Year',
    value_name='Quantity_000Tonne'
)
df_qty['Year'] = df_qty['Year'].str.replace('Quantity ', '').str.strip()

# Melt Value
df_val = pd.melt(
    df_states,
    id_vars=['States'],
    value_vars=val_cols,
    var_name='Year',
    value_name='Value_INR'
)
df_val['Year'] = df_val['Year'].str.replace('Value ', '').str.replace(r'\(P\)', '', regex=True).str.strip()


df_clean = pd.merge(df_qty, df_val, on=['States', 'Year'])

In [9]:
# Step 4: String standardization and type casting
df_clean.rename(columns={'States': 'State'}, inplace=True)

# Remove special footnote character '*' from Telangana
df_clean['State'] = df_clean['State'].str.replace('*', '', regex=False).str.strip()

# Standardize historical name 'Orissa' to official 'Odisha'
df_clean['State'] = df_clean['State'].replace({'Orissa': 'Odisha'})

# Handle 'NA' strings, coerce to numeric, and fill empty cells with 0
df_clean['Quantity_000Tonne'] = pd.to_numeric(df_clean['Quantity_000Tonne'].replace('NA', 0), errors='coerce').fillna(0)
df_clean['Value_INR'] = pd.to_numeric(df_clean['Value_INR'].replace('NA', 0), errors='coerce').fillna(0)

# Flag data status based on preliminary fiscal year
df_clean['Data_Status'] = df_clean['Year'].apply(lambda y: 'Provisional' if '2014-15' in y else 'Actual')

In [10]:
# Step 5: Post-cleaning verification & sanity check
print("\n--- Cleaned Data Sample ---")
print(df_clean.head(10))

print("\n--- Unique Cleaned States ---")
print(sorted(df_clean['State'].unique()))

print("\n--- Total Verified Coal Production ('000 Tonnes) ---")
print(f"{df_clean['Quantity_000Tonne'].sum():,.2f}")

# Export for SQL pipeline
df_clean.to_csv('coal_cleaned.csv', index=False)
print("\nPipeline execution complete: 'coal_cleaned.csv' generated successfully.")


--- Cleaned Data Sample ---
               State      Unit     Year  Quantity_000Tonne    Value_INR  \
0     Andhra Pradesh  000tonne  2010-11            51333.0   81106100.0   
1  Arunachal Pradesh  000tonne  2010-11              299.0    1106000.0   
2              Assam  000tonne  2010-11             1101.0    4072600.0   
3       Chhattisgarh  000tonne  2010-11           113824.0   58256200.0   
4    Jammu & Kashmir  000tonne  2010-11               24.0      22400.0   
5          Jharkhand  000tonne  2010-11           108949.0  185716200.0   
6          Meghalaya  000tonne  2010-11             6974.0   25796800.0   
7     Madhya Pradesh  000tonne  2010-11            71104.0   93673600.0   
8        Maharashtra  000tonne  2010-11            39336.0   53628800.0   
9             Odisha  000tonne  2010-11           102565.0   73545300.0   

  Data_Status  
0      Actual  
1      Actual  
2      Actual  
3      Actual  
4      Actual  
5      Actual  
6      Actual  
7      Actual  
8